# Class 1 — Introduction to Machine Learning & Understanding Supervised Learning




## Agenda

1. AI vs ML vs DL — a rigorous, non-marketing definition
2. Taxonomy of ML: Supervised, Unsupervised, Semi-supervised, Self-supervised, Reinforcement
3. Real-world applications mapped to ML paradigms
4. Supervised Learning — formal mathematical setup
5. Regression vs Classification — geometry, loss functions, output spaces
6. Problem framing — the single most undervalued skill in industry ML
7. Production considerations & training–serving skew (preview)
8. FAANG-style interview Q&A
9. Mini case study: framing a real business problem end-to-end


## 1. Why This Topic Matters

### Importance in Machine Learning
This is the *axiomatic layer* of ML. Every advanced topic — gradient boosting, transformers, RLHF — is a specialization of the supervised / unsupervised / RL trichotomy introduced here. Misunderstanding the taxonomy leads to choosing the wrong loss, the wrong evaluation metric, and ultimately the wrong model class.

### Importance in AI Systems
Modern AI systems (LLMs, recommenders, fraud engines) are *compositions* of supervised, self-supervised, and RL stages. GPT-style models are pre-trained with self-supervised next-token prediction, fine-tuned with supervised SFT, and aligned via RLHF. You cannot reason about such pipelines without owning the taxonomy cold.

### Importance in Production ML
Problem framing — deciding whether a business problem is regression, classification, ranking, or anomaly detection — determines the data contract, label collection cost, monitoring strategy, and SLA. Reframing a problem from regression to classification (e.g. "predict revenue" → "predict revenue bucket") often eliminates months of label noise.

### Importance in Interviews
Every ML system design round at FAANG begins with: *"How would you frame this problem?"* Candidates who jump to model choice without framing fail. Strong candidates spend 5–10 minutes on framing, labels, and evaluation before mentioning a model.

### Real-World Relevance
- **Netflix**: ranking (learning-to-rank, a supervised variant) — not classification.
- **Uber ETA**: regression with quantile loss, not MSE — because tails matter for user trust.
- **Stripe Radar**: binary classification with extreme class imbalance — framing as anomaly detection is *wrong* and degrades recall.
- **ChatGPT**: self-supervised pre-training + supervised fine-tuning + RLHF — three paradigms in one product.


## 2. Learning Outcomes

By the end of this class learners will be able to:

1. Precisely distinguish AI, ML, and DL with set-theoretic clarity.
2. Classify any real-world problem into the correct ML paradigm.
3. Formalize a supervised learning problem as $(\mathcal{X}, \mathcal{Y}, \mathcal{D}, \mathcal{H}, \mathcal{L})$.
4. Derive the empirical risk minimization (ERM) objective from first principles.
5. Choose between regression and classification based on output space and business loss.
6. Identify when a problem looks supervised but is actually a ranking, ordinal, or survival problem.
7. Frame a fuzzy business problem (e.g. "reduce churn") into a precise ML problem statement.
8. Articulate the bias–variance and label-cost tradeoffs implicit in problem framing.
9. Explain training–serving skew and how framing decisions cause it.
10. Identify leakage at the framing stage (using future-information features).
11. Write production-grade Python scaffolding for a supervised pipeline using `scikit-learn`.
12. Defend framing choices in an interview using structured reasoning (data → label → loss → metric → constraint).
13. Recognize when *not* to use ML at all (rule-based baselines, heuristics).
14. Map evaluation metrics to business KPIs.
15. Anticipate distribution shift implications of the chosen framing.


## 3. Deep Conceptual Notes

### 3.1 AI vs ML vs DL — Set-Theoretic View

$$
\text{DL} \subsetneq \text{ML} \subsetneq \text{AI}
$$

- **AI**: any system that exhibits goal-directed behavior — includes rule-based expert systems, search (A*), constraint solvers, and ML.
- **ML**: systems that *improve performance on a task $T$ with experience $E$ measured by $P$* (Mitchell, 1997). The key: parameters are learned from data, not programmed.
- **DL**: ML using deep, hierarchical, differentiable function approximators (typically neural nets with $\geq 2$ hidden layers) optimized via backprop.

A chess engine using alpha-beta pruning is AI but not ML. Linear regression is ML but not DL. A transformer is all three.

### 3.2 Taxonomy of ML

| Paradigm | Data | Goal | Canonical example |
|---|---|---|---|
| Supervised | $(x_i, y_i)$ | Learn $f: \mathcal{X} \to \mathcal{Y}$ | Spam detection |
| Unsupervised | $x_i$ only | Discover structure | K-means, PCA |
| Semi-supervised | Small labeled + large unlabeled | Leverage unlabeled data | Medical imaging |
| Self-supervised | $x_i$, labels derived from $x_i$ | Pretext task → useful representations | BERT, SimCLR, GPT |
| Reinforcement | $(s_t, a_t, r_t, s_{t+1})$ | Maximize expected return | AlphaGo, RLHF |

**Self-supervised is not unsupervised.** SSL fabricates supervised labels from the data itself (mask a token, predict it). This distinction matters in interviews.

### 3.3 Formal Setup of Supervised Learning

Let $\mathcal{X}$ be the input space and $\mathcal{Y}$ the output space. Assume there exists an unknown joint distribution $\mathcal{D}$ over $\mathcal{X} \times \mathcal{Y}$. We observe i.i.d. samples:

$$
S = \{(x_i, y_i)\}_{i=1}^{n} \sim \mathcal{D}^n
$$

Given a hypothesis class $\mathcal{H}$ (e.g. all linear functions) and a loss $\ell: \mathcal{Y} \times \mathcal{Y} \to \mathbb{R}_{\geq 0}$, define the **true risk**:

$$
R(h) = \mathbb{E}_{(x,y) \sim \mathcal{D}}\big[\ell(h(x), y)\big]
$$

Since $\mathcal{D}$ is unknown, we minimize the **empirical risk**:

$$
\hat{R}_S(h) = \frac{1}{n} \sum_{i=1}^{n} \ell(h(x_i), y_i)
$$

**ERM principle**: $\hat{h} = \arg\min_{h \in \mathcal{H}} \hat{R}_S(h)$.

The entire supervised ML edifice — linear regression, SVMs, GBMs, transformers — is ERM with different $(\mathcal{H}, \ell)$ pairs and different optimizers.

### 3.4 Regression vs Classification

|  | Regression | Classification |
|---|---|---|
| $\mathcal{Y}$ | $\mathbb{R}$ (or $\mathbb{R}^d$) | Finite set $\{c_1, \dots, c_K\}$ |
| Typical loss | MSE, MAE, Huber, Quantile | Cross-entropy, Hinge |
| Decision boundary | n/a | Hypersurface in $\mathcal{X}$ |
| Output calibration | Point estimate or distribution | Probability mass |
| Failure mode | Heavy tails dominate MSE | Class imbalance dominates accuracy |

**Subtle case — Ordinal regression**: $\mathcal{Y} = \{1,2,3,4,5\}$ (star ratings). Treating it as classification ignores order; treating it as regression assumes equal spacing. Correct framing uses ordinal loss (e.g. cumulative link models).

### 3.5 Problem Framing — The Real Skill

A framing checklist (use this in every interview):

1. **Business objective** — what KPI moves?
2. **Prediction target** — what exactly is $y$? At what granularity (user, session, request)?
3. **Label source** — who/what generates $y$? Cost? Latency? Noise?
4. **Feature availability at serving time** — any feature unavailable at inference is a leakage trap.
5. **Loss function** — does it match the asymmetric business cost? (False negative in fraud ≠ false positive.)
6. **Evaluation metric** — offline metric correlated with online KPI?
7. **Latency / throughput / cost constraint** — rules out model classes.
8. **Feedback loop** — does the model's predictions change future data? (Recommenders always do.)

### 3.6 Worked Example — Framing "Reduce Churn"

Naïve framing: "Predict if user will churn." Problems:
- Churn horizon undefined (7d? 30d? 90d?). Each gives a different model.
- "Churn" definition undefined (no login? no purchase? cancel subscription?).
- If we use last-7-days-activity as a feature, we leak: at serving time we predict for users who *just* became inactive.

Correct framing:
- $y$ = "user makes zero purchases in days $[t+1, t+30]$"
- $x$ = features computed strictly from days $[t-90, t]$
- Loss: weighted log-loss with $w_+ = $ expected LTV (saving a high-LTV churner > saving a low-LTV one)
- Metric: PR-AUC (imbalanced) + uplift on a holdout treated with a retention campaign


## 4. Production-Grade Code Walkthrough

We will build a minimal but production-styled supervised pipeline and observe the difference between regression and classification on the same dataset (California Housing — regression, then binarized → classification).


In [ ]:
# Cell 1 — environment
import sys, sklearn, numpy as np, pandas as pd
print("python:", sys.version.split()[0], "| sklearn:", sklearn.__version__)
np.random.seed(42)

python: 3.12.13 | sklearn: 1.6.1


In [ ]:
# Cell 2 — load data
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target  # y in 100k USD units
print(X.shape, y.describe().round(3).to_dict())

(20640, 8) {'count': 20640.0, 'mean': 2.069, 'std': 1.154, 'min': 0.15, '25%': 1.196, '50%': 1.797, '75%': 2.647, 'max': 5.0}


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

reg = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

reg.fit(X_tr, y_tr)

pred = reg.predict(X_te)

rmse = np.sqrt(mean_squared_error(y_te, pred))
mae = mean_absolute_error(y_te, pred)
r2 = r2_score(y_te, pred)

print(f"RMSE={rmse:.3f}  MAE={mae:.3f}  R2={r2:.3f}")

RMSE=0.746  MAE=0.533  R2=0.576


In [ ]:
# Cell 4 — CLASSIFICATION framing (same data, binarized at median)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

thresh = y.median()
yc = (y > thresh).astype(int)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X, yc, test_size=0.2, random_state=42, stratify=yc)
clf = Pipeline([("scale", StandardScaler()),
                ("model", LogisticRegression(max_iter=1000))]).fit(Xc_tr, yc_tr)
proba = clf.predict_proba(Xc_te)[:, 1]
print(classification_report(yc_te, (proba > 0.5).astype(int), digits=3))
print(f"ROC-AUC = {roc_auc_score(yc_te, proba):.3f}")

              precision    recall  f1-score   support

           0      0.829     0.835     0.832      2065
           1      0.834     0.827     0.831      2063

    accuracy                          0.831      4128
   macro avg      0.831     0.831     0.831      4128
weighted avg      0.831     0.831     0.831      4128

ROC-AUC = 0.913


In [ ]:
# Cell 5 — illustrate framing decision via business cost
# Suppose under-predicting a high-value house costs 2x more than over-predicting.
# Asymmetric loss can be approximated with quantile regression.
from sklearn.ensemble import GradientBoostingRegressor
q90 = GradientBoostingRegressor(loss="quantile", alpha=0.9, n_estimators=200,
                                max_depth=3, random_state=42).fit(X_tr, y_tr)
q10 = GradientBoostingRegressor(loss="quantile", alpha=0.1, n_estimators=200,
                                max_depth=3, random_state=42).fit(X_tr, y_tr)
pi = pd.DataFrame({"y": y_te.values, "lo": q10.predict(X_te), "hi": q90.predict(X_te)})
coverage = ((pi.y >= pi.lo) & (pi.y <= pi.hi)).mean()
print(f"Empirical 80% prediction-interval coverage = {coverage:.3f}")

Empirical 80% prediction-interval coverage = 0.759


## 5. Tradeoffs & Pitfalls

- **Premature modeling**: choosing XGBoost before framing — interview red flag.
- **Label leakage at framing time**: defining $y$ using information not available at inference.
- **Wrong loss**: using MSE when the business cost is asymmetric (e.g. ETAs).
- **Wrong granularity**: predicting per-user when the decision is per-session.
- **Ignoring base rate**: 99% accuracy on a 99/1 imbalance means *nothing*.
- **Not using ML when a rule wins**: if `if amount > $10000 and country in blacklist` catches 80% of fraud at near-zero cost, ship that first.

## 6. Production Considerations

- **Training–serving skew**: features computed in batch (Spark) vs realtime (Flink) — must be byte-identical. Use a feature store (Feast, Tecton, Vertex FS).
- **Label latency**: in churn, label takes 30 days to materialize. Model trains on stale labels — plan for it.
- **Feedback loops**: a fraud model that blocks transactions never sees the "would-have-been-fraud" outcome → biased retraining set. Mitigate with exploration (random allow-through on a small slice).
- **Monitoring**: track input drift (PSI, KS), prediction drift, and *delayed* outcome metrics separately.


## 7. FAANG-Style Interview Q&A

**Q1. Define ML formally. Why is ERM not enough?**
ERM minimizes empirical risk but the quantity we care about is true risk. The generalization gap $R(h) - \hat{R}_S(h)$ can be bounded (VC dimension, Rademacher complexity), motivating regularization, cross-validation, and structural risk minimization.

**Q2. A PM asks: "Build an ML model to improve user engagement." Walk through framing.**
Clarify: which engagement metric (DAU, session length, retention)? Decision granularity (user/session/notification)? Available labels? Feedback loop? Constraints (latency, fairness)? Propose 2 framings (e.g. CTR classification vs dwell-time regression) with tradeoffs.

**Q3. When would you choose classification over regression even though the target is numeric?**
When (a) business decisions are bucketed (e.g. risk tiers), (b) label noise on exact values is huge but bucket assignment is reliable, (c) the loss is naturally 0/1 over a threshold, or (d) calibrated probabilities are more useful than point estimates.

**Q4. What's the difference between self-supervised and unsupervised learning?**
SSL constructs supervised labels from the input itself via a pretext task (masked LM, contrastive views). Loss is supervised in form. Unsupervised has no labels — only structure discovery (clustering, density estimation).

**Q5. Give an example where ML is the wrong solution.**
Low-volume, high-stakes decisions with crisp rules (e.g. tax computation). ML adds noise, opacity, and maintenance cost with no accuracy gain over deterministic logic.

**Q6. What is training–serving skew and how do framing decisions cause it?**
Skew = feature distribution at inference ≠ training. Caused by (a) different feature pipelines online vs offline, (b) features that depend on future info offline but not online, (c) label definitions that drift. Framing decisions that rely on features not robustly available at serving time create skew.

**Q7. Why is accuracy a bad metric for fraud detection?**
Severe class imbalance (~0.1% fraud) → predicting "no fraud" always yields 99.9% accuracy with zero recall. Use PR-AUC, recall@k%FPR, or expected monetary loss.

**Q8. You have $10^9$ unlabeled rows and $10^4$ labeled rows. Approach?**
Self-supervised pretraining on the $10^9$ to learn representations, then supervised fine-tuning on the $10^4$. Or semi-supervised methods (pseudo-labeling, consistency regularization, FixMatch).


## 8. Mini Case Study — Frame "Reduce 30-day churn for a SaaS product"

- **Step 1 — KPI**: 30-day retention rate.
- **Step 2 — Target**: $y_u = \mathbb{1}[\text{user } u \text{ has zero active sessions in days } [t+1, t+30]]$.
- **Step 3 — Features**: aggregates from $[t-90, t]$ only — session counts, feature usage, support tickets, plan tier.
- **Step 4 — Loss**: weighted log-loss with weight $\propto$ user MRR.
- **Step 5 — Offline metric**: PR-AUC + calibration (Brier).
- **Step 6 — Online metric**: uplift in 30-day retention vs control in an A/B retention campaign.
- **Step 7 — Constraints**: nightly batch scoring acceptable, no realtime requirement.
- **Step 8 — Baseline**: heuristic rule "no login in 14 days" — must beat it.

